In [ ]:
import sys
from pathlib import Path
module_path = str(Path('..').resolve())
if module_path not in sys.path:
    print(f"added {module_path} to path")
    sys.path.append(module_path)
for line in sys.path:
    print(line)

import pyfar as pf
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from algorithms.common import chunked, make_spectrogram, map_signal, draw_panning_histogram
from algorithms.avendano import ambiance_index, panning_index
from algorithms.gomez import panning_histogram
from algorithms.tzanetakis import stereo_panning_features
from algorithms import gomez, sarroff, tardieu
import librosa
from scipy.signal import ShortTimeFFT, get_window
import matplot2tikz
from python_auditory_toolbox import auditory_toolbox as pat
import scipy
plt.rcParams['font.family'] = "Times"
# plt.rcParams['text.usetex'] = True

%load_ext jupyter_tikz
%load_ext autoreload
%autoreload 2

In [ ]:
def to_dat(X, Y):
    return '\n'.join([f"{x} {y}" for x,y in zip(X,Y)])

In [ ]:
hrirs, sources = pf.signals.files.head_related_impulse_responses(position='horizontal', diffuse_field_compensation=True)

In [ ]:
elevation = 0
azimuth_L = np.pi/3 # 60 degrees left
azimuth_R = -np.pi/3 # 60 degrees right
radius = 1

desired_direction_L = pf.Coordinates.from_spherical_elevation( azimuth_L, elevation, radius)
desired_direction_R = pf.Coordinates.from_spherical_elevation( azimuth_R, elevation, radius)

index_L, _ = sources.find_nearest(desired_direction_L)
index_R, _ = sources.find_nearest(desired_direction_R)

sources.show([index_L, index_R])
plt.show()


In [ ]:
ax = pf.plot.freq_group_delay(hrirs[index_R], label=["Left ear", "Right ear"])
# ax[0].legend()
# ax[1].legend()
plt.show()

In [ ]:
kick = pf.io.read_audio(Path.home() / "Music/freesound/581451__johnnydekk__fractanimal_acoustic_drum_kit_kick_2.wav")
hihat = pf.io.read_audio(Path.home() / "Music/freesound/581473__johnnydekk__fractanimal_acoustic_drum_kit_high_hat_3.wav")
# ax_kick = pf.plot.time_freq(kick)
ax_hihat = pf.plot.time_freq(hihat)
plt.show()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(8,4), sharex=True, layout='tight')



def render(audio, hrir, ax, slug):
    binaural_audio = pf.dsp.convolve(audio, hrir)
    cropped = pf.dsp.time_crop(binaural_audio, [0,0.03], unit='s')
    cropped.time[0] += 1
    cropped.time[1] -= 1
    cropped.sampling_rate /= 1000
    # cropped.times *= 1000
    X = (cropped.times * 1000)

    print("ITD", X[np.argmax(cropped.time, axis=1)])
    print("MAX", np.max(cropped.time, axis=1))
    print("MIN", np.min(cropped.time, axis=1))

    Path(f"../../papers/kappa/figures/{slug}_L.dat").write_text(to_dat(X, cropped.time[0]))
    Path(f"../../papers/kappa/figures/{slug}_R.dat").write_text(to_dat(X, cropped.time[1]))

    pf.plot.time(cropped, label=["Left ear", "Right ear"], ax=ax, color='k')
    ax.set_ylim(-2,2)

# def latex_plot_hihat():
    # txt = '\\begin{tikzpicture}'

    # txt += '\\end{tikzpicture}'

    

render(hihat, hrirs[index_R], ax[0], "hihat")
render(kick, hrirs[index_R], ax[1], "kick")

ax[0].set_xlabel(None)
ax[1].set_xlabel("Time (ms)")
ax[0].set_ylabel("Hi-hat")
ax[1].set_ylabel("Kick drum")
plt.show()

tikz.save(
    "../../papers/kappa/tests/matplot2tikz.tex", 
    figure=fig, 
    axis_width='\\textwidth', 
    axis_height='4cm',
)

In [ ]:
music = pf.io.read_audio("/Users/richard/Music/archive/09. Tsubasao Kudasai.flac", stop=int(10. * 44100))
pf.plot.spectrogram(music, freq_scale='log')
plt.show()

In [ ]:
from matplotlib import colormaps, color_sequences

# def amplitude_pan(sig: pf.Signal, pan: float) -> pf.Signal:
#     assert -1 <= pan <= 1.
#     assert sig.cshape == (2,)
#     a = (pan + 1) / 2
#     al, ar = np.cos(a * 0.5 * np.pi), np.sin(a * 0.5 * np.pi)
#     print(a, al, ar)
#     left, right = sig.time[0] * al, sig.time[1] * ar
#     return pf.Signal(np.stack([left, right]), sig.sampling_rate)

# def downmix(sig: pf.Signal) -> pf.Signal:
#     assert sig.cshape == (2,)
#     a = 1/np.sqrt(2)
#     data = np.sum(sig.time, axis=0) * a
#     return pf.Signal(np.stack([data, data]), sig.sampling_rate)

# def downmix_to_mono(sig: pf.Signal) -> pf.Signal:
#     assert sig.cshape == (2,)
#     a = 1/np.sqrt(2)
#     data = np.sum(sig.time, axis=0) * a
#     return pf.Signal(data, sig.sampling_rate)


# def make_panned_sound(pan: float = 0.5):
#     fvocals = "/Users/richard/Music/freesound/245325__vocalpercussion__vocal_percussion_deh_deh_boom.wav"
#     fguitar = "/Users/richard/Music/freesound/474398__bangcorrupt__blues_guitar_intro.flac"
#     ftrumpet = "/Users/richard/Music/freesound/2323__synapse__trumpetloop01.aiff"

#     def _load(f):
#         audio = pf.io.read_audio(f)
#         if audio.sampling_rate != 48000:
#             audio = pf.dsp.resample(audio, sampling_rate=48000)
#         audio = pf.Signal(audio.time[0, :480000], sampling_rate=audio.sampling_rate)
#         audio = pf.dsp.normalize(audio, reference_method='energy', target=0.5)
#         return audio
    
#     vocals = _load(fvocals)
#     guitar = _load(fguitar)
#     trumpet = _load(ftrumpet)
#     # sound1 = amplitude_pan(downmix(pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f1), sampling_rate=48000), [0.,10.], unit='s')), -pan)
#     # sound2 = amplitude_pan(downmix(pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f2), sampling_rate=48000), [0.,10.], unit='s')), pan)
#     # sound = sound1 + sound2
#     print(vocals, guitar, trumpet)
#     # left = vocals * 0.5 + guitar * 0.7 + trumpet * 0.1
#     # right = vocals * 0.5 + guitar * 0.3 + trumpet * 0.9
#     left = guitar * 0.7 + trumpet * 0.1
#     right = guitar * 0.3 + trumpet * 0.9
#     sound = pf.Signal(np.stack([left.time[0], right.time[0]]), 48000)
#     return sound

def get_music():
    # f = "/Users/richard/Music/archive/09. Tsubasao Kudasai.flac"
    # return pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f), sampling_rate=48000), [0,480000])
    f = "/Users/richard/Music/projects/panning_and_reverb/panning_and_reverb_wav.wav"
    return pf.io.read_audio(f)

sound = get_music()

# freqs, times, spect = pf.dsp.spectrogram(sound)
fft_size = 1024
hop_size = 512
stft = ShortTimeFFT(
    get_window("hann", fft_size),
    hop_size,
    sound.sampling_rate
)

spect = stft.stft(sound.time)
spect = np.swapaxes(spect, 1, 2)
times = np.arange(spect.shape[1]) * (stft.hop / stft.fs)
freqs = stft.f

mosaic = """
aaaaaaaaaaaaaaaaaaaaaaaaaa.
aaaaaaaaaaaaaaaaaaaaaaaaaa.
bbbbbbbbbbbbbbbbbbbbbbbbbbB
bbbbbbbbbbbbbbbbbbbbbbbbbbB
bbbbbbbbbbbbbbbbbbbbbbbbbbB
ccccccccccccccccccccccccccC
ccccccccccccccccccccccccccC
ccccccccccccccccccccccccccC
"""
fig, ax = plt.subplot_mosaic(mosaic, figsize=(8,5), layout='tight')
# fig = plt.figure()
# ax = dict()
# ax["a"] = fig.add_subplot(3, 1, 1)
# ax["b"] = fig.add_subplot(3, 1, 2, sharex=ax["a"])
# ax["c"] = fig.add_subplot(3, 1, 3, sharex=ax["a"])

ai = ambiance_index(spect, lamb=0.85)
pi = panning_index(spect)


def decibels(x, mindb=-70.):
    epsilon = np.nextafter(np.float64(0), np.float64(1))
    # epsilon = 1e-16
    return np.clip(20 * np.log10(x + epsilon), a_min=mindb, a_max=float("inf"))

def make_peaks(sound, pps):
    n_sections = int((sound.n_samples * pps) / sound.sampling_rate)
    channel_peaks = []
    for channel in sound.time:
        chunks = np.array_split(channel, n_sections)
        chunks[-2] = np.append(chunks[-2], chunks[-1])
        del chunks[-1]
        channel_peaks.append([np.max(np.abs(chunk)) for chunk in chunks])
    return pf.Signal(np.stack(channel_peaks), pps)

def plot_stereo(sound: pf.Signal, pps: int, ax: plt.Axes, colormap: str = 'berlin'):
    peaks = make_peaks(sound, pps)
    offset = np.max(peaks.time)
    x = np.arange(peaks.n_samples) / peaks.sampling_rate
    color_l, color_r = colormaps.get(colormap)([0.1,0.9])
    ax.fill_between(x, peaks.time[0] + offset, -peaks.time[0] + offset, color=color_l)
    im = ax.fill_between(x, peaks.time[1] - offset, -peaks.time[1] - offset, color=color_r)
    ax.set_yticks([offset, -offset], labels=["left", "right"])
    ax.set_xlim(x[[0,-1]])
    ax.set_ylim([-0.8,0.8])
    ax.set_xticks([])
    return im


weights = np.sum(np.abs(spect), axis=0)
panogram = np.vstack([np.histogram(pi[i], bins=100, range=[-1,1], weights=weights[i])[0] for i in range(len(pi))])

plot_stereo(sound, 400, ax['a'])
# ax['a'].set_title("Audio waveform")
# ax['a'].set_xlabel("Time (s)")

extent = (times[0], times[-1], freqs[0], freqs[-1])
display_ai = np.nan_to_num(decibels(np.rot90(ai)))
display_pi = np.rot90(pi)
display_panogram = decibels(np.rot90(panogram))

# print(display_ai.min(), display_ai.max())
# print(display_pi.min(), display_pi.max())


img_ai = ax['b'].imshow(display_ai, aspect='auto', cmap='grey', extent=extent)
ax['b'].set_ylabel("Frequency (Hz)")
ax['b'].set_yticks([0,5000,10000,15000,20000], labels=["0", "5k", "10k", "15k", "20k"])
# ax['b'].set_xlabel("Time (s)")
# ax['b'].set_title("Ambience Index", y=1.0, x=0, pad=-14)

ax['b'].set_xticks([])


img_pi = ax['c'].imshow(display_pi, aspect='auto', cmap='berlin', extent=extent)
ax['c'].set_ylabel("Frequency (Hz)")
ax['c'].set_yticks([0,5000,10000,15000,20000], labels=["0", "5k", "10k", "15k", "20k"])
ax['c'].set_xlabel("Time (s)")
# ax['c'].set_title("Panning Index", y=1.0, x=0, pad=-14)

cbar = fig.colorbar(img_ai, cax=ax["B"])
cbar.ax.set_yticks([display_ai.min(), display_ai.max()], labels=[0,1])


cbar = fig.colorbar(img_pi, cax=ax["C"], ticks=[display_pi.min(), display_pi.max()])
cbar.ax.set_yticks([display_pi.min(), 0, display_pi.max()], labels=["Left", "Centre", "Right"])



props = dict(boxstyle='round', facecolor='white', alpha=0.8)
props2 = dict(boxstyle='round', facecolor='white', alpha=0.9)
x = 0.01
y = 0.94
ax["a"].text(x, y, "Audio Waveform", transform=ax["a"].transAxes, verticalalignment='top', bbox=props)
ax["b"].text(x, y, "Ambience Index", transform=ax["b"].transAxes, verticalalignment='top', bbox=props2)
ax["c"].text(x, y, "Panning Index", transform=ax["c"].transAxes, verticalalignment='top', bbox=props2)

fig.savefig("../../papers/kappa/figures/waveforms.pdf")

Audio(sound.time, rate=sound.sampling_rate)

In [ ]:
new_length_400pps = int((sound.n_samples * 400) / sound.sampling_rate)
new_length_10pps = int((sound.n_samples * 10) / sound.sampling_rate)
new_length_1pps = int(sound.n_samples / sound.sampling_rate)

print(sound.n_samples, new_length_400pps, new_length_10pps, new_length_1pps)

def make_peaks(sound, pps):
    n_sections = int((sound.n_samples * pps) / sound.sampling_rate)
    channel_peaks = []
    for channel in sound.time:
        chunks = np.array_split(channel, n_sections)
        chunks[-2] = np.append(chunks[-2], chunks[-1])
        del chunks[-1]
        channel_peaks.append([np.max(np.abs(chunk)) for chunk in chunks])
    return pf.Signal(np.stack(channel_peaks), pps)

def plot_stereo(sound: pf.Signal, pps: int, ax: plt.Axes, colormap: str = 'berlin'):
    peaks = make_peaks(sound, pps)
    offset = np.max(peaks.time)
    x = np.arange(peaks.n_samples) / peaks.sampling_rate
    color_l, color_r = colormaps.get(colormap)([0.1,0.9])
    ax.fill_between(x, peaks.time[0] + offset, -peaks.time[0] + offset, color=color_l)
    ax.fill_between(x, peaks.time[1] - offset, -peaks.time[1] - offset, color=color_r)
    ax.set_yticks([])


fig, ax = plt.subplots(3,1, figsize=(10, 10))

plot_stereo(sound, 400, ax[0])
plot_stereo(sound, 10, ax[1])
plot_stereo(sound, 1, ax[2])



In [ ]:
nx, ny = (3,4)
idx = np.mgrid[:nx,:ny].reshape(2, -1).T
frame = [64,256,1024,4096]
hop = [0.5, 0.75, 0.875]


s = 3
fig, ax = plt.subplots(ny, nx, figsize=(nx*s, ny*s), sharex=True)

# file = "659528__johaynes__jos-genedrums-150bpm-42.wav"
# file = "581473__johnnydekk__fractanimal_acoustic_drum_kit_high_hat_3.wav"
file = "245325__vocalpercussion__vocal_percussion_deh_deh_boom.wav"
transient_signal = downmix_to_mono(pf.io.read_audio(f"/Users/richard/Music/freesound/{file}"))
transient_signal = pf.dsp.time_crop(transient_signal, [0.85,1.1], unit='s')

print(transient_signal)
for i,j in idx:
    f = frame[j]
    h = hop[i]
    a = ax[j,i]
    pf.plot.spectrogram(transient_signal, window_length=f, window_overlap_fct=h, ax=a, colorbar=False)
    a.set_title(f"{f}, {int(f*(1-h))}")
    a.set_xlabel(None)
    a.set_ylabel(None)

fig.savefig("spectrogram_accuracy.pdf")

In [ ]:
import librosa
audio, sr = librosa.load(librosa.util.example('trumpet'))
fig, ax = plt.subplots(2,1)

D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
S = librosa.feature.melspectrogram(S=D, sr=sr, n_mels=32)

librosa.display.specshow(data=D, sr=sr, ax=ax[0], y_axis='linear')
librosa.display.specshow(S, sr=sr, ax=ax[1], y_axis='mel')

fig.savefig("mel.pdf")


In [ ]:


fig, ax = plt.subplots(3,1, figsize=(8, 8), sharex=True)

# audio, sr = librosa.load(librosa.util.example('trumpet', hq=True))
# signal = pf.signals.exponential_sweep_time(48000, [20,20000], sampling_rate=48000)
signal = pf.io.read_audio("/Users/richard/Music/freesound/31912__acclivity__globalwarming_marg.wav")
audio, sr = signal.time[0], signal.sampling_rate

start, end = 0.8, 2.6
duration = end - start
audio = audio[int(start * sr):int(end * sr)]

stft_bands = 2048
num_bands = 32

def kilo_formatter(x):
    if x < 1000:
        return str(int(x))
    else:
        return f"{int(x/1000)}k"


linear_freqs = np.fft.rfftfreq(stft_bands, d=1/sr)
erb_freqs = pat.ErbSpace(low_freq=100, high_freq=sr/2, n=num_bands)[::-1]

coefs = pat.MakeErbFilters(sr, num_bands, low_freq=100)
y = pat.ErbFilterBank(audio, coefs)
y *= np.sign(y) # half wave rectify
y = np.sqrt(y) # square root

# low pass
fc = 800
b, a = scipy.signal.butter(4, fc/sr, btype='low', analog=False)
y = scipy.signal.filtfilt(b, a, y)

coch_chunk_length = 0.02
chunk_size = int(coch_chunk_length * sr)
hop_size = int(0.5 * chunk_size)
chunks = np.lib.stride_tricks.sliding_window_view(y, chunk_size, axis=1)[:, ::hop_size, :]
print(chunks.shape)
# coch = np.sqrt(np.sum(np.pow(chunks, 2), axis=(2,))) # RMS
coch = np.mean(chunks, axis=2)


D = librosa.amplitude_to_db(np.abs(librosa.stft(audio, n_fft=stft_bands)), ref=np.max)
S = librosa.feature.melspectrogram(S=D, sr=sr, n_mels=num_bands)
mel_freqs = librosa.mel_frequencies(n_mels=num_bands, fmax=sr/2)
print(D.shape, S.shape, coch.shape)


images = []

xmax = D.shape[1] - 1
images.append(ax[0].imshow(np.flipud(D), aspect='auto', cmap='magma', extent=(0, duration, 0, linear_freqs[-1])))
images.append(ax[1].imshow(np.flipud(S), aspect='auto', cmap='magma', extent=(0, duration, 0, num_bands)))
images.append(ax[2].imshow(librosa.amplitude_to_db(coch), aspect='auto', cmap='magma', extent=(0, duration, 0, num_bands)))


ax[0].set_ylabel("Frequency (Hz)")
ax[0].set_yticks([0,5000,10000,15000,20000], labels=["0", "5k", "10k", "15k", "20k"])

desired_ticks = [100, 500, 1000, 2000, 5000, 10000, 20000]

ax[1].set_ylabel("Frequency (Hz)")
yticks = np.interp(desired_ticks, mel_freqs, np.arange(num_bands))
print(yticks)
ax[1].set_yticks(yticks, labels=[kilo_formatter(x) for x in desired_ticks])

ax[2].set_ylabel("Frequency (Hz)")
yticks = np.interp(desired_ticks, erb_freqs, np.arange(num_bands))
ax[2].set_yticks(yticks, labels=[kilo_formatter(x) for x in desired_ticks])
ax[2].set_xlabel("Time (s)")

props = dict(boxstyle='round', facecolor='white', alpha=0.9)
x = 0.02
y = 0.94
ax[0].text(x, y, "Spectrogram", transform=ax[0].transAxes, verticalalignment='top', bbox=props)
ax[1].text(x, y, "Mel Spectrogram", transform=ax[1].transAxes, verticalalignment='top', bbox=props)
ax[2].text(x, y, "Cochleagram", transform=ax[2].transAxes, verticalalignment='top', bbox=props)


fig.colorbar(images[0], ax=ax, label="Sound intensity (dB)")
fig.savefig("../../papers/kappa/figures/spect_comp.pdf")

plt.show()

In [ ]:
fs = 44100
fft_length = 4096

fmin = 20
num_bands = 8

freqs = np.fft.rfftfreq(fft_length, d=1/fs)
F = np.tile(freqs, (num_bands,1))
print(F.shape)

mel_filters = librosa.filters.mel(sr=fs, n_fft=fft_length, n_mels=num_bands, fmin=fmin)
mel_filters /= np.max(mel_filters)
# mel_filters[mel_filters == 0] = np.nan # showing all the zero weights looks cluttered, so just get rid of them


fig, ax = plt.subplots(1,2, figsize=(8,3))
ax[0].plot(F.T, mel_filters.T, color='k')
ax[1].semilogx(F.T, mel_filters.T, color='k')


tikz_fig = r"""
\begin{tikzpicture}

\definecolor{darkgray176}{RGB}{176,176,176}

\begin{groupplot}[
    group style={
        group size=2 by 1,
        horizontal sep=10pt,
    },
    xlabel={Frequency (Hz)},
]

\nextgroupplot[
    xmin=-2000,
    xmax=25000,
    xtick={0, 5000, 10000, 15000, 20000},
    xticklabels={{0}, {5k}, {10k}, {15k}, {20k}},
    scaled x ticks=false,
    ylabel={Filter weights},
    title={Mel filterbank (linear frequency)},
]
PLOT1
LABELS1

\nextgroupplot[
    xmode=log,
    log basis x={10},
    xmin=15,
    xmax=50000,
    xtick={20, 200, 2000, 20000},
    xticklabels={{20}, {200}, {2k}, {20k}},
    yticklabels=\empty,
    title={Mel filterbank (logarithmic frequency)},
]
PLOT2
LABELS2

\end{groupplot}

\end{tikzpicture}
"""

def to_table(X, Y):
    table = ["table {%"]
    for x,y in zip(X, Y):
        if y>0:
            table.append(f"{x:.3f} {y:.3f}")
    table.append("}")
    return table

def to_plot(table):
    table_txt = '\n'.join(table)
    return f"\\addplot [black]\n{table_txt};"


# tikz_fig = matplot2tikz.get_tikz_code(fig)
# print(tikz_fig)

plot1 = []
plot2 = []
labels1 = []
labels2 = []

for n, filt in enumerate(mel_filters):
    x = F[0][np.argmax(filt)]
    y = np.max(filt)
    print(n, x, y)
    lin_anchor = "south west" if n in [1,2] else "south"
    log_anchor = "south"
    lin_label = f"\\node[draw=black,circle,fill=white] at (axis cs:{x:.3f},{y:.3f}) [anchor={lin_anchor}, inner sep=2pt, outer sep=4pt] {{{n}}};"
    log_label = f"\\node[draw=black,circle,fill=white] at (axis cs:{x:.3f},{y:.3f}) [anchor={log_anchor}, inner sep=2pt, outer sep=4pt] {{{n}}};"
    # labels1.append(lin_label)
    # labels2.append(log_label)

    plot = to_plot(to_table(F[0], filt))
    plot1.append(plot)
    plot2.append(plot)

    # ax[0].text(x, y, f"n={n}", transform=ax[0].transData)
    # ax[1].text(x, y, f"n={n}", transform=ax[1].transAxes)
    # ax[0].text(1, 0, f"fuck", transform=ax[0].transData)

# labels = "\n".join(labels)
tikz_fig = tikz_fig.replace("PLOT1", "\n".join(plot1))
tikz_fig = tikz_fig.replace("PLOT2", "\n".join(plot2))
tikz_fig = tikz_fig.replace("LABELS1", "\n".join(labels1))
tikz_fig = tikz_fig.replace("LABELS2", "\n".join(labels2))
# tikz_fig.insert(tikz_fig.find(), labels)

# print(plot1[0])

preamble = r"""
\usepackage{tikz}
\usepackage{pgfplots}
\usetikzlibrary{calc,arrows.meta}
\usepgfplotslibrary{groupplots}
\pgfplotsset{compat=1.18}
"""

Path("../../papers/kappa/figures/mel_filterbank.tikz").write_text(tikz_fig)

# %tikz -e -t=pgfplots -l="calc,arrows.meta" -lp="groupplots" -p="$preamble" "$tikz_fig"
%tikz -e -p "$preamble" "$tikz_fig"

# fig.savefig("test.pdf")

# plt.show()


# mel_filters = librosa.amplitude_to_db(mel_filters) # display in decibles to make it similar to the gammatone filters
# print(librosa.filters.mel_frequencies(n_mels=8, fmax=24000, fmin=fmin))

# fig, ax = plt.subplots(2,2, figsize=(8, 6), sharey=True)


# ax[0,0].plot(F.T, mel_filters.T, color='k')
# ax[0,0].set_xscale('log')
# ax[0,0].set_ylabel("Amplitude")
# ax[0,0].set_xlabel("Frequency (Hz)")
# ax[0,0].set_title("Mel filters (logarithmic)")
# ax[0,1].plot(F.T, mel_filters.T, color='k')
# ax[0,1].set_xscale('linear')
# # ax[1].set_ylabel("Amplitude")
# ax[0,1].set_xlabel("Frequency (Hz)")
# ax[0,1].set_title("Mel filters (linear)")

# ax[0,0].set_ylim([-60, 0])
# ax[0,0].set_xlim(20, fs/2)
# ax[0,1].set_ylim([-60, 0])
# ax[0,1].set_xlim(20, fs/2)

# fig.tight_layout()
# fig.savefig("mel_filters.pdf")

# print(mel_filters.shape, freqs.shape)

# # for i in range(len(mel_filters)):
#     # Path(f"../../papers/kappa/figures/mel_{i}.dat").write_text(to_dat(freqs, mel_filters[i]))



# # Now plot gammatone filters
# fcoefs = pat.MakeErbFilters(fs, 8, low_freq=fmin)
# impulse_len = fft_length
# x = np.zeros(impulse_len)
# x[0] = 1
# y = pat.ErbFilterBank(x, fcoefs)
# take_len = impulse_len // 2
# resp = 20*np.log10(np.abs(np.fft.fft(y.T, axis=0)))[:take_len]
# ax[1,0].semilogx(np.arange(take_len)*(fs/impulse_len),resp, color='k')
# ax[1,0].set_xlim(20, fs/2)
# ax[1,0].set_ylim(-60, 0)
# ax[1,0].set_ylabel("Filter response (dB)")
# ax[1,0].set_xlabel("Frequency (Hz)")


# gammatone_freq = np.arange(take_len)*(fs/impulse_len)
# for i in range(resp.shape[1]):
#     mask = resp[:, i] >= -60
#     Path(f"../../papers/kappa/figures/gammatone_{i}.dat").write_text(to_dat(gammatone_freq[mask], resp[:,i][mask]))



In [ ]:
import scipy
a = 1.99
rv = scipy.stats.gamma(a)
x = np.linspace(0, 10, 200)
y = rv.pdf(x) * np.cos(8*x) - 0.5
y2 = rv.pdf(x-1) * np.cos(8*(x-1)) + 0.5
y3 = (y*0.5) + 0.75
plt.plot(x, y)
plt.plot(x, y2)
plt.plot(x, y3)

def write_path(x, y):
    coords = " ".join([f"({a[0]:.3f},{a[1]:.3f})" for a in  zip(x,y)])
    return f"\\addplot [domain=0:10, no markers, smooth] coordinates {{{coords}}};"

Path("itd_R.tex").write_text(write_path(x,y))
Path("itd_L.tex").write_text(write_path(x,y2))
Path("ild_R.tex").write_text(write_path(x,y))
Path("ild_L.tex").write_text(write_path(x,y3))

print(max(rv.pdf(x)))


In [ ]:
def plot_hrir():
    elevation = 0
    azimuth_L = np.pi/6
    azimuth_R = -azimuth_L
    radius = 1

    desired_direction_L = pf.Coordinates.from_spherical_elevation( azimuth_L, elevation, radius)
    desired_direction_R = pf.Coordinates.from_spherical_elevation( azimuth_R, elevation, radius)
    index_L, _ = sources.find_nearest(desired_direction_L)
    index_R, _ = sources.find_nearest(desired_direction_R)
    print(desired_direction_L.azimuth, desired_direction_R.azimuth)


    dur = 64
    start = 10
    sig = pf.dsp.time_crop(hrirs[index_R], [start,start+dur])
    pf.plot.time(sig)

    X = np.arange(dur)
    Y = sig.time

    def txt(Y, name):
        points = "--".join([f"({x:.3f},{y:.3f})" for x,y in zip(X,Y)])
        return f"\\path[local bounding box={name}, draw=black] {points};"
    
    # print(txt(Y[0]))
    # print(txt(Y[1]))

    Path("../../papers/kappa/figures/brir_L.tex").write_text(txt(Y[0], "left_brir"))
    Path("../../papers/kappa/figures/brir_R.tex").write_text(txt(Y[1], "right_brir"))

plot_hrir()

In [ ]:
# def spiral(theta, a, b):
#     r = a * np.exp(b * theta)
#     x = np.cos(r)
#     y = np.sin(r)
#     return x, y

def cart2pol(xy):
    rho = np.sqrt(xy.real**2 + xy.imag**2)
    phi = np.arctan2(xy.imag, xy.real)
    return np.array([rho, phi])

def pol2cart(rp):
    rho, phi = rp
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    return x + y * 1j

def spiral(a, b, t):
    print(t.shape)
    spir = a * np.exp((b + 1j) * t)
    spir.real *= -1
    return spir
    # rho, phi = cart2pol(spir)
    # # phi += np.pi/2
    # return pol2cart([rho, phi])

def coord(xy):
    return f"({xy.real:.3f},{xy.imag:.3f})"

def plot_cochlea(t, a=1, b=0.08, tension=0.618):
    d = 0.18
    inner = spiral(a, b, t)
    outer1 = spiral(a * (1+d), b, t)
    outer2 = spiral(a * (1-d), b, t)

    freqs = [200, 400, 800, 1600, 3200, 6400, 12000, 20000]
    log_pos = np.log10(freqs)
    log_pos -= min(log_pos)
    log_pos /= max(log_pos)
    log_pos_t = np.interp(log_pos, np.linspace(-0.01, 1.01, len(t)), t)

    base_label = spiral(a, b, t[-1:] + 0.168)[0]
    apex_label = inner[0] + 0.618 - 0.2j

    basilar_membrane_label1 = spiral(a, b, t[-4:-3])[0]
    basilar_membrane_label2 = pol2cart(cart2pol(basilar_membrane_label1) + [1.618, 0])

    frq_marks = spiral(a, b, log_pos_t)

    def path(XY, colour):
        points = " ".join([f"({x:.3f},{y:.3f})" for x,y in zip(XY.real, XY.imag)])
        return f"\\draw[{colour}] plot [smooth, tension={tension}] coordinates {{ {points} }};"

    def txt():
        txt = "\\begin{tikzpicture}"
        txt += path(inner, "black, dashed")
        txt += path(outer1, "black, thick")
        txt += path(outer2, "black, thick")
        txt += f"\\draw[black, thick] {coord(outer1[0])} to[bend right=90, looseness=1.618] {coord(outer2[0])};"
        txt += f"\\draw[black, thick] {coord(outer1[-1])} to[bend left=90, looseness=1] {coord(inner[-1])};"
        txt += f"\\draw[black, thick] {coord(outer2[-1])} to[bend right=90, looseness=1] {coord(inner[-1])};"

        txt += f"\\node[] (base_label) at {coord(base_label)} {{Base}};"
        txt += f"\\node[] (apex_label) at {coord(apex_label)} {{Apex}};"
        txt += f"\\draw[latex-] {coord(basilar_membrane_label1)} -- {coord(basilar_membrane_label2)};"
        txt += f"\\node[anchor=north] (basilar_membrane_label) at {coord(basilar_membrane_label2)} {{Basilar membrane}};"
        for pos,frq in zip(frq_marks, freqs):
            txt += f"\\node[circle, fill=black, draw=black, minimum size=4pt, text width=0pt, inner sep=0] (point_{frq}) at {coord(pos)} {{}};"
            txt += f"\\node[anchor=south, fill=white, draw=black, outer sep=4pt] (point_{frq}) at {coord(pos)} {{{frq}}};"

        txt += "\\end{tikzpicture}"
        return txt.replace('\\', '\n\\')

    return txt()

drawing = plot_cochlea(np.linspace(0, 18, 50))
# print(drawing)
Path("../../papers/kappa/figures/cochlea.tex").write_text(drawing)


%tikz -e "$drawing"
    


In [ ]:
# f = "/Users/richard/Music/projects/panning_and_reverb/panning_and_reverb_wav.wav"
f = "/Users/richard/Music/archive/09. Tsubasao Kudasai.flac"
music = pf.dsp.time_crop(pf.io.read_audio(f), [0, 30], unit='s')

chunks = chunked(music, 1024, 512)
spect = np.fft.rfft(chunks)
print(spect.shape)

def __stats(x):
    print(f"{np.min(x)=} {np.max(x)=} {np.mean(x)=} {np.std(x)=}")


hist = panning_histogram(spect, 100)

plt.plot(hist)
plt.show()

In [ ]:
number_of_turns = 3
max_angle = number_of_turns * 2 * np.pi
t = np.linspace(0, 1, 100)
starting_angle = -np.pi/2
theta = starting_angle + (t * max_angle)
k = 1.8
r1 = 1.4
r = r1 * np.exp(-k * t)
x1 = 0
y1 = 0
X = x1 + r * np.cos(theta)
Y = y1 + r * np.sin(theta)


normals = pol2cart((1, theta + (np.pi/2)))
X2 = X + normals.real
Y2 = Y + normals.imag


minf = 20
maxf = 20000
minx = np.log10(minf)
maxx = np.log10(maxf)
linx = t * (maxx - minx) + minx
# print(linx)


# fig, ax = plt.subplots(1,1, figsize=(12,6))
thickness = np.geomspace(0.35, 0.06, 100)
norm1 = pol2cart((thickness, theta))
norm2 = pol2cart((-thickness, theta))
X1 = X + norm1.real
Y1 = Y + norm1.imag
X2 = X + norm2.real
Y2 = Y + norm2.imag

X *= -1
X1 *= -1
X2 *= -1

def line_length(x, y):
    deltax = np.diff(x)
    deltay = np.diff(y)
    return np.sum(np.hypot(deltax, deltay))


l = line_length(X, Y)
major_ticks = np.linspace(-l, 0, 4)

def to_points(x, y):
    if hasattr(x, '__iter__'):
        return " ".join([f"({xx:.3f},{yy:.3f})" for xx,yy in zip(x,y)])
    else:
        return f"({x:.3f},{y:.3f})"


tikz = r"""
\begin{tikzpicture}
\begin{axis}[
    xmode=log,  
    ymax=2,
    ymin=-0.5,
    xmin=10, 
    xmax=50000, 
    width=15cm, 
    height=6cm,
    axis x line*=bottom,
    axis y line=none,
    xlabel={Frequency (Hz)},
    xtick={20, 200, 2000, 20000},
    xticklabels={20, 200, 2k, 20k}
]

    \coordinate (base) at (axis cs:20000,0);
    \coordinate (apex) at (axis cs:20,0);
    \coordinate (base_upper) at (axis cs:20000,0.2);
    \coordinate (base_lower) at (axis cs:20000,-0.2);
    \coordinate (apex_upper) at (axis cs:20,0.1);
    \coordinate (apex_lower) at (axis cs:20,-0.1);

    \draw[black] (base_lower) to[bend right=90, looseness=1.618] (base);
    \draw[black] (base) to[bend right=90, looseness=1.618] (base_upper);
    \draw[black] (apex_lower) to[bend left=90, looseness=1.618] (apex_upper);
    \draw[black] (base_upper) -- (apex_upper);
    \draw[black] (base_lower) -- (apex_lower);
    \draw[black, dashed] (base) -- (apex);

    \draw[black,-Latex] (axis cs: 2000, 0.7) node[inner sep=5pt, anchor=south] {Basilar membrane} -- (axis cs: 2000, 0);
\end{axis}

\coordinate (shift) at ($(base) + (0,GOUP)$);
\begin{scope}[shift={(shift)}]
\draw[black, densely dotted] plot[smooth, tension=0.618] coordinates {SPIRAL_UPPER};
\draw[black, densely dotted] plot[smooth, tension=0.618] coordinates {SPIRAL_LOWER};
\draw[black, densely dotted] SPIRAL_APEX1 to[bend left=90, looseness=1.618] SPIRAL_APEX2;
\end{scope}

\node[black] at (base) [inner sep=10pt, anchor=west] {Base};
\node[black] at (apex) [inner sep=10pt, anchor=east] {Apex};

\end{tikzpicture}
"""

tikz = tikz.replace("GOUP", str(r1))
tikz = tikz.replace("SPIRAL_UPPER", to_points(X1, Y1))
# tikz = tikz.replace("SPIRAL_INNER", to_points(X, Y))
tikz = tikz.replace("SPIRAL_LOWER", to_points(X2, Y2))
tikz = tikz.replace("SPIRAL_APEX1", to_points(X1[-1], Y1[-1]))
tikz = tikz.replace("SPIRAL_APEX2", to_points(X2[-1], Y2[-1]))

print(tikz)
Path("../../papers/kappa/figures/unfurled_cochlea.tikz").write_text(tikz)

%tikz -e -t=pgfplots -l="calc,arrows.meta" "$tikz"




In [ ]:
from algorithms.tardieu import make_subbands, extract_envelope, make_cochleagram
import rdp

audio = pf.signals.exponential_sweep_time(4410, [100, 20000])
# audio = pf.io.read_audio("/Users/richard/Music/freesound/31912__acclivity__globalwarming_marg.wav")
# audio = pf.dsp.time_crop(audio, [0.8, 2.6], unit='s')
# audio.time *= 6
subbands = make_subbands(audio, 8)[0] # shape (channel, subband, time)
envelope = extract_envelope(subbands, audio.sampling_rate) # shape (subband, time)
coch = make_cochleagram(envelope, 1024, 512)
freqs = np.round(pat.ErbSpace(low_freq=100, high_freq=44100/2, n=8)[::-1]).astype(int)
print(freqs)


def _reduce(x, y, max_mse):
    epsilon = 0.
    M = np.stack([x, y]).T
    epsilon = np.geomspace(0.00001, 0.001, 20)
    for e in epsilon:
        xp, yp = rdp.rdp(M, epsilon=e).T
        yr = np.interp(x, xp, yp)
        err = np.mean(np.pow(y - yr, 2))
        # print(e, len(xp), err)
        if err >= max_mse:
            return xp, yp

def _plot_subbands(ax, times, signal, reduce=False):
    N = signal.shape[0]
        
    for n, sig in enumerate(np.flip(signal, axis=0)):
        if reduce:
            red_times, red_sig = _reduce(times, sig, 1e-6)
            print(f"reduced {len(times)} to {len(red_times)} points")
            ax.plot(red_times, red_sig + n, color='k', linewidth=0.3)
        else:
            ax.plot(times, sig + n, color='k', linewidth=0.3)


    # img = ax.plot(times, np.flip(signal, axis=0).T + np.arange(len(signal) * separation), color='k', linewidth=0.3)
    ax.set_yticks(np.arange(N), labels=np.arange(N))
    ax.set_xticks([0., 0.02, 0.04, 0.06, 0.08, 0.1], labels=[0,20,40,60,80,100])
    ax.set_xlabel("Time (ms)")


# Plotting all audio points on tiny graphs breaks a lot of stuff.
# Downsample audio for display
# def _downsample(x: NDArray[np.float64]) -> NDArray[np.float64]:
#     # q = 2
#     # n = 0
#     # for _ in range(n):
#     #     x = scipy.signal.decimate(x, q, axis=1)
#     # x = x[:,::32]
#     times = np.linspace(0,1,x.shape[1])
#     return times, x



fig, ax = plt.subplots(1,3, figsize=(8,3), sharey=True, layout='constrained')

X = audio.times
reduce = False
_plot_subbands(ax[0], X, subbands, reduce=reduce)
_plot_subbands(ax[1], X, envelope, reduce=reduce)
# _plot_subbands(ax[2], np.arange(coch.shape[1]), coch)
ax[2].imshow(coch, aspect='auto', extent = (0, 100, -0.5, 7.5), cmap='magma')
ax[2].set_yticks(np.arange(8), np.arange(8))

ax[0].set_title("Gammatone sub-bands")
ax[0].set_ylabel("Frequency (Hz)")
ax[0].set_yticklabels(freqs)
ax[1].set_title("Envelopes")
ax[2].set_title("Cochleagram")

for a in ax:
    a.set_xlabel("Time (ms)")
    a.set_ylim((-0.5, 8))

def _make_data(times, signal, downsample=None):
    altered_data = np.flip(signal, axis=0) + np.expand_dims(np.arange(signal.shape[0]), axis=0).T
    all_data = np.stack([times, *altered_data]).T
    if downsample:
        return all_data[::downsample]
    return all_data


fig.savefig("../../papers/kappa/figures/cochleagram_steps.pdf")

# matplot2tikz.save("../../papers/kappa/figures/cochleagram_steps.tikz", figure=fig)

# np.savetxt("../../papers/kappa/figures/subband_data.dat", _make_data(audio.times, subbands), fmt="%.3e")
# np.savetxt("../../papers/kappa/figures/envelope_data.dat", _make_data(audio.times, envelope, downsample=8), fmt="%.3e")
# plt.imsave("../../papers/kappa/figures/cochleagram.png", coch)

np.savez("test_subbands.npz", time=audio.times, data=subbands)
np.savez("test_envelopes.npz", time=audio.times, data=envelope)


In [ ]:
matplot2tikz.save("../../papers/kappa/tests/cochleagram_steps.tikz", figure=fig)

In [ ]:
from IPython.display import Audio
from algorithms.sarroff import frequency_azimuth_analysis_signal
from algorithms.common import make_spectrogram, map_signal

sr = 48000
fft_size = 128
hop_size = 64
gain = 0.1

# a = pf.signals.files.guitar(sampling_rate=sr)
# b = pf.signals.files.speech(sampling_rate=sr)
# b = pf.dsp.pad_zeros(b, a.n_samples - b.n_samples)


partials_a = np.fft.rfftfreq(fft_size, d=1/sr)[[11,21,31,41]]
partials_b = np.fft.rfftfreq(fft_size, d=1/sr)[[15,25,35,45]]
a = sum([pf.signals.sine(f, sr, sampling_rate=sr, amplitude=gain) for f in partials_a])
b = sum([pf.signals.sine(f, sr, sampling_rate=sr, amplitude=gain) for f in partials_b])

a = a * [1,0.2]
b = b * [0.2,1]
mix = a + b


# mix = pf.io.read_audio("/Users/richard/Music/projects/panning_and_reverb/panning_and_reverb_wav.wav")


spect = np.abs(make_spectrogram(mix, fft_size=fft_size, hop_size=hop_size))
faas = frequency_azimuth_analysis_signal(spect, 100)
print(faas.shape)
fap = faas[20]
display(Audio(mix.time, rate=mix.sampling_rate))
xx, yy = np.meshgrid(np.fft.rfftfreq(fft_size, d=1/sr), np.arange(fap.shape[1]))
print(xx.shape, yy.shape, fap.shape)

fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(projection='3d')
ax.plot_wireframe(xx, yy, fap.T, color='k', linewidth=0.5)
ax.set_xlabel("Frequency")
ax.set_ylabel("Azimuth")
ax.set_zlabel("Peak Magnitude")
ax.xaxis.set_inverted(True)
ax.set_yticks([0,100,200], labels=["L", "C", "R"])
fig.savefig("../../papers/kappa/figures/adress.pdf")
plt.show()


# panning_spect = np.argmax(faas, axis=-1)
# fig = plt.figure(figsize=(12.8,7.2))
# ax = fig.add_subplot()
# print(np.max(panning_spect))
# ax.imshow(np.rot90(panning_spect), aspect='auto', cmap='berlin', vmin=0, vmax=200)
# plt.show()

In [ ]:
def _display_spf(spf, title, ax, ylim=None):
    mm = spf[0:4]
    sm = spf[4:8]
    ms = spf[8:12]
    ss = spf[12:16]
    xl = np.arange(4) + (0.618 / 2)
    xr = np.arange(4) + (0.618)
    ax.errorbar(xl, mm, yerr=ms, fmt='_', color='k', label='Slow panning')
    ax.errorbar(xr, sm, yerr=ss, fmt='x', color='k', label='Fast panning')
    ax.set_ylim(bottom=-0.1)
    ax.set_title(title)
    ax.set_ylim(ylim)
    ax.set_xlim(0, 4)
    ax.set_xticks([0.5,1.5,2.5,3.5], ["Full", "Low", "Mid", "High"])
    # ax.tick_params("x", rotation=45)
    ax.set_xlabel("Frequency band")

tracks = [
    ("/Users/richard/Music/fma_small/123/123899.mp3", "123899"),
    # ("/Users/richard/Music/fma_small/113/113513.mp3", "A Crooked Pulse\nDistorted Course"),
    ("/Users/richard/Music/fma_small/117/117614.mp3", "117614"),
    ("/Users/richard/Music/fma_small/122/122809.mp3", "122809"),
]

fig, axes = plt.subplots(1,3, figsize=(7,3), sharey=True, layout='tight')
for (filepath, track), ax in zip(tracks, axes):
    audio = pf.io.read_audio(filepath)
    spect = make_spectrogram(audio)
    spf = stereo_panning_features(spect, audio.sampling_rate, 40)
    _display_spf(spf, f"{track}", ax, [-0.1, 6.7])
axes[0].legend(loc='upper left')
axes[0].set_ylabel("Feature value")

fig.savefig("../../papers/kappa/figures/spf.pdf")

In [ ]:
tracks = [
    ("/Users/richard/Music/fma_small/123/123899.mp3", "123899"),
    ("/Users/richard/Music/fma_small/117/117614.mp3", "117614"),
    ("/Users/richard/Music/fma_small/122/122809.mp3", "122809"),
]

fig, axes = plt.subplots(1,3, figsize=(7,3), layout='tight')
for (filepath, track), ax in zip(tracks, axes):
    audio = pf.io.read_audio(filepath)
    spect = make_spectrogram(audio)
    ph_gomez = gomez.panning_histogram(spect, 65)
    ph_sarroff = sarroff.panning_histogram(np.abs(spect), 32)
    ph_sarroff /= np.sum(ph_sarroff)
    ymax = np.max([ph_sarroff, ph_gomez])
    ylim = (ymax * -0.1, ymax * 1.1)
    draw_panning_histogram(ph_gomez, ax, Path(filepath).stem, ylim, color='k', linestyle='--', label='Gomez')
    draw_panning_histogram(ph_sarroff, ax, Path(filepath).stem, ylim, color='k', linestyle='-', label='Sarroff')

for ax in axes[1:3]:
    ax.set_ylabel(None)
    ax.get_legend().remove()

fig.savefig("../../papers/kappa/figures/panning_histograms.pdf")

In [ ]:
tracks = [
    "/Users/richard/Music/fma_small/123/123899.mp3",
    "/Users/richard/Music/fma_small/117/117614.mp3",
    "/Users/richard/Music/fma_small/122/122809.mp3",
]

def _make_asf(filepath):
    audio = pf.io.read_audio(filepath)
    coch = tardieu.cochleagram(audio, 0.04, 0.02, 70)
    return tardieu.amplitude_stereo_features(coch)

all_asf = [_make_asf(f) for f in tracks]


In [ ]:
fig, axes = plt.subplots(1,3, figsize=(7,3), sharey=True, layout='constrained')
for filepath, asf, ax in zip(tracks, all_asf, axes):
    freqs = pat.ErbSpace(low_freq=30, high_freq=44100/2, n=70)[::-1]
    ax.errorbar(asf.mean_over_time, freqs, xerr=asf.std_over_time, marker='o', linestyle='none', color='k', linewidth=0.5, markersize=2)
    ax.set_yscale('log')
    ax.set_ylim(20, 30000)
    ax.set_xlim(-0.11, 0.11)
    ax.set_xlabel("Panning amount")
    ax.set_title(Path(filepath).stem)
axes[0].set_ylabel("Frequency (Hz)")
fig.savefig("../../papers/kappa/figures/asf.pdf")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource

# 1. Define grid parameters for the smooth surface mesh
N = 100  
theta_steps = np.linspace(np.pi / 2, 0, N + 1)
phi_steps = np.linspace(0, np.pi / 2, N + 1)
THETA, PHI = np.meshgrid(theta_steps, phi_steps)

# 2. Map surface to Cartesian coordinates
X = np.sin(THETA) * np.cos(PHI)
Y = np.sin(THETA) * np.sin(PHI)
Z = np.cos(THETA)

# 3. Define the true symmetric Surface Center Point in polar coordinates
c_val = 1.0 / np.sqrt(3)  # Coordinates where X = Y = Z
t_center = np.arccos(c_val)
p_center = np.pi / 4

# 4. Define the buffer distance from the corners
chord_dist = 0.25
offset = 1.006  # Floats lines slightly above the surface mesh
sweep_res = 50

# 5. FIX: Calculate exact polar stopping coordinates to precisely hit the arcs
# --- Z-Corner intercept logic
z_stop_val = 1.0 - (chord_dist**2 / 2.0)
t_stop_z = np.arccos(z_stop_val)
p_stop_z = np.pi / 4

# --- X-Corner intercept logic (where X meets the chord distance, and Y=Z by symmetry)
x_stop_val = 1.0 - (chord_dist**2 / 2.0)
yz_stop_val = np.sqrt((1.0 - x_stop_val**2) / 2.0)
t_stop_x = np.arccos(yz_stop_val)
p_stop_x = np.arctan2(yz_stop_val, x_stop_val)

# --- Y-Corner intercept logic (where Y meets the chord distance, and X=Z by symmetry)
y_stop_val = 1.0 - (chord_dist**2 / 2.0)
xz_stop_val = np.sqrt((1.0 - y_stop_val**2) / 2.0)
t_stop_y = np.arccos(xz_stop_val)
p_stop_y = np.arctan2(y_stop_val, xz_stop_val)

# 6. FIX: Revert main paths to pure polar linear interpolation (Great-Circle tracks)
def polar_to_cart(t, p, r_offset=1.006):
    """
    Transforms 3D spherical polar coordinates into Cartesian coordinates 
    for a sphere of unit radius (R=1).

    Parameters:
    -----------
    t : float or numpy.ndarray
        Theta (polar angle / colatitude) in radians. Represents the angular 
        displacement down from the positive Z-axis [0, pi].
    p : float or numpy.ndarray
        Phi (azimuthal angle) in radians. Represents the angular rotation 
        counter-clockwise from the positive X-axis in the XY plane [0, 2*pi].
    r_offset : float, optional
        A fractional multiplier applied to the base unit radius (R=1). 
        Defaults to 1.006. This scales the vector out slightly to prevent 
        3D rendering artifacts ("Z-fighting") when plotting lines directly 
        on top of a pre-rendered surface mesh.

    Returns:
    --------
    tuple of (float or numpy.ndarray)
        A 3-element tuple containing the mapped (X, Y, Z) Cartesian coordinates.
        Matches the dimension and shape of the input array arrays.
    """
    x = np.sin(t) * np.cos(p) * r_offset
    y = np.sin(t) * np.sin(p) * r_offset
    z = np.cos(t) * r_offset
    return x, y, z

def draw_text_on_surface(ax, text, t, p, text_offset=1.05, **kwargs):
    """
    Plots text labels directly onto the 3D surface of the sphere using polar coordinates.

    Parameters:
    -----------
    ax : matplotlib.axes._subplots.Axes3DSubplot
        The active 3D plot axis object.
    text : str
        The text string to display.
    t : float
        Theta (polar angle / colatitude) in radians.
    p : float
        Phi (azimuthal angle) in radians.
    text_offset : float, optional
        The radial scaling factor to float the text cleanly above the surface 
        and prevent it from clipping through the 3D mesh. Defaults to 1.05.
    **kwargs : dict, optional
        Standard matplotlib text properties (e.g., color, fontsize, fontweight, 
        horizontalalignment).

    Returns:
    --------
    matplotlib.text.Text3D
        The created 3D text object.
    """
    # Convert polar angles to 3D Cartesian space with a safe text offset
    x, y, z = polar_to_cart(t, p, r_offset=text_offset)
    
    # Set default clean alignments if they aren't explicitly provided
    kwargs.setdefault('horizontalalignment', 'center')
    kwargs.setdefault('verticalalignment', 'center')
    kwargs.setdefault('color', 'black')
    
    # Render text onto the 3D axes
    return ax.text(x, y, z, text, **kwargs)

# Interpolate angles linearly and convert to 3D paths
x_main = polar_to_cart(np.linspace(t_center, t_stop_x, 50), np.linspace(p_center, p_stop_x, 50), offset)
y_main = polar_to_cart(np.linspace(t_center, t_stop_y, 50), np.linspace(p_center, p_stop_y, 50), offset)
z_main = polar_to_cart(np.linspace(t_center, t_stop_z, 50), np.linspace(p_center, p_stop_z, 50), offset)

# 7. Keep the distance-based boundary arcs from scratch
sweep = np.linspace(0, np.pi / 2, sweep_res)

# --- Z-Corner Arc (Constant distance from (0,0,1))
z_arc_z = np.full_like(sweep, z_stop_val)
r_xy_z = np.sqrt(1.0 - z_arc_z**2)
z_b = polar_to_cart(np.arccos(z_arc_z), sweep, offset)

# --- X-Corner Arc (Constant distance from (1,0,0))
x_arc_x = np.full_like(sweep, x_stop_val)
r_yz_x = np.sqrt(1.0 - x_arc_x**2)
x_b = np.array([x_arc_x, r_yz_x * np.cos(sweep), r_yz_x * np.sin(sweep)]) * offset

# --- Y-Corner Arc (Constant distance from (0,1,0))
y_arc_y = np.full_like(sweep, y_stop_val)
r_xz_y = np.sqrt(1.0 - y_arc_y**2)
y_b = np.array([r_xz_y * np.cos(sweep), y_arc_y, r_xz_y * np.sin(sweep)]) * offset

# 8. Plot setup
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Smooth surface shading
light = LightSource(azdeg=315, altdeg=45)
rgb = light.shade(Z, cmap=plt.get_cmap('grey'), blend_mode='soft')
ax.plot_surface(X, Y, Z, facecolors=rgb, edgecolor='none', linewidth=0, antialiased=False, shade=False)

# Plot Surface-Hugging Main Paths (Black)
ax.plot(*x_main, color='red', linewidth=2, zorder=5)
ax.plot(*y_main, color='red', linewidth=2, zorder=5)
ax.plot(*z_main, color='red', linewidth=2, zorder=5)

# Plot Distance-Perfect Boundary Lines (Crimson Red)
ax.plot(*x_b, color='red', linewidth=2, zorder=6)
ax.plot(*y_b, color='red', linewidth=2, zorder=6)
ax.plot(*z_b, color='red', linewidth=2, zorder=6)

# Highlight Center Point
# ax.scatter(*polar_to_cart(t_center, p_center, offset), color='cyan', s=100, zorder=7)

draw_text_on_surface(ax, "I", 1.45, 0.2, color='white', zorder=10)
draw_text_on_surface(ax, "II", np.pi/4, 0.5, color='black', zorder=10)
draw_text_on_surface(ax, "III", np.pi/4, np.pi/2 - 0.5, color='black', zorder=10)
draw_text_on_surface(ax, "IV", np.pi/2 - 0.35, np.pi/4, color='white', zorder=10)
draw_text_on_surface(ax, "V", np.pi/2 - 0.1, np.pi/2 - 0.1, color='white', zorder=10)
draw_text_on_surface(ax, "VI", 0.2, np.pi/4 + 0.2, color='black', zorder=10)

print(polar_to_cart(1., 1, 2))

# Visual styling
ax.set_ylabel(r'variance $\sigma_p$')
ax.set_xlabel(r'variance $\sigma_s$')
ax.set_zlabel(r'variance $\sigma_{uv}$')
ax.set_xlim(0, 1.1)
ax.set_ylim(0, 1.1)
ax.set_zlim(0, 1.1)
ax.view_init(elev=15, azim=60)

fig.savefig("../../papers/kappa/figures/harma.pdf")


In [ ]:
from algorithms import ziemer

fig, axes = plt.subplots(2,3, figsize=(7,4), sharex=True, layout='constrained')

box_feats = []
corr_feats = []

for filepath, col in zip(tracks, axes.T):
    audio = pf.io.read_audio(filepath)
    box, corr = ziemer.feature_vectors(audio, 0.04, 0.04)
    t = np.arange(len(box)) * 0.04
    col[0].plot(t, box, color='k', linewidth=0.3)
    col[1].plot(t, corr, color='k', linewidth=0.3)
    col[0].set_ylim(-0.03,0.43)
    col[1].set_ylim(-1.1,1.1)
    col[1].set_xlabel("Time (s)")
    box_feats.append(box)
    corr_feats.append(corr)


axes[0,0].set_ylabel("Normalized box count")
axes[1,0].set_ylabel("Channel correlation")
axes[0,1].set_yticklabels([])
axes[0,2].set_yticklabels([])
axes[1,1].set_yticklabels([])
axes[1,2].set_yticklabels([])
axes[0,0].set_title(Path(tracks[0]).stem)
axes[0,1].set_title(Path(tracks[1]).stem)
axes[0,2].set_title(Path(tracks[2]).stem)

fig.savefig("../../papers/kappa/figures/goniometer_features.pdf")


In [ ]:
from matplotlib.collections import LineCollection
import math
from numpy import pi

from algorithms.common import polar_to_cartesian

def colored_line(x, y, c, ax=None, **lc_kwargs):
    """
    Plot a line with a color specified along the line by a third value.

    It does this by creating a collection of line segments. Each line segment is
    made up of two straight lines each connecting the current (x, y) point to the
    midpoints of the lines connecting the current point with its two neighbors.
    This creates a smooth line with no gaps between the line segments.

    Parameters
    ----------
    x, y : array-like
        The horizontal and vertical coordinates of the data points.
    c : array-like
        The color values, which should be the same size as x and y.
    ax : matplotlib.axes.Axes, optional
        The axes to plot on. If not provided, the current axes will be used.
    **lc_kwargs
        Any additional arguments to pass to matplotlib.collections.LineCollection
        constructor. This should not include the array keyword argument because
        that is set to the color argument. If provided, it will be overridden.

    Returns
    -------
    matplotlib.collections.LineCollection
        The generated line collection representing the colored line.
    """

    xy = np.stack((x, y), axis=-1)
    xy_mid = np.concat(
        (xy[0, :][None, :], (xy[:-1, :] + xy[1:, :]) / 2, xy[-1, :][None, :]), axis=0
    )
    segments = np.stack((xy_mid[:-1, :], xy, xy_mid[1:, :]), axis=-2)
    # Note that
    # segments[0, :, :] is [xy[0, :], xy[0, :], (xy[0, :] + xy[1, :]) / 2]
    # segments[i, :, :] is [(xy[i - 1, :] + xy[i, :]) / 2, xy[i, :],
    #     (xy[i, :] + xy[i + 1, :]) / 2] if i not in {0, len(x) - 1}
    # segments[-1, :, :] is [(xy[-2, :] + xy[-1, :]) / 2, xy[-1, :], xy[-1, :]]

    # lc_kwargs["array"] = c
    lc_kwargs["alpha"] = c
    lc = LineCollection(segments, **lc_kwargs)

    # Plot the line collection to the axes
    ax = ax or plt.gca()
    ax.add_collection(lc)

    return lc


def rotate_matrix (x, y, angle, x_shift=0, y_shift=0, units="DEGREES"):
    """
    Rotates a point in the xy-plane counterclockwise through an angle about the origin
    https://en.wikipedia.org/wiki/Rotation_matrix
    :param x: x coordinate
    :param y: y coordinate
    :param x_shift: x-axis shift from origin (0, 0)
    :param y_shift: y-axis shift from origin (0, 0)
    :param angle: The rotation angle in degrees
    :param units: DEGREES (default) or RADIANS
    :return: Tuple of rotated x and y
    """

    # Shift to origin (0,0)
    x = x - x_shift
    y = y - y_shift

    # Convert degrees to radians
    if units == "DEGREES":
        angle = math.radians(angle)

    # Rotation matrix multiplication to get rotated x & y
    xr = (x * math.cos(angle)) - (y * math.sin(angle)) + x_shift
    yr = (x * math.sin(angle)) + (y * math.cos(angle)) + y_shift

    return xr, yr

fig, ax = plt.subplots(1,2, figsize=(6,2.8), layout='constrained')
start = 19.48
audio = pf.dsp.time_crop(pf.io.read_audio(tracks[1]), [start, start + .04], unit='s')
x, y = rotate_matrix(audio.time[0], audio.time[1], 45)
c = np.linspace(0, 1, audio.n_samples)
colored_line(x, y, c, ax=ax[0], color='k', linewidth=0.5)
ax[0].set_xlim(-1,1)
ax[0].set_ylim(-1,1)

N = 10
xq = (x * N).astype(int) + N
yq = (y * N).astype(int) + N
grid = np.zeros((2*N, 2*N))
for xx, yy in np.stack([xq,yq]).T:
    grid[xx,yy] += 1
unique_boxes = np.sum(grid != 0)
grid[grid==0] = np.nan
cmap = plt.cm.plasma
cmap.set_bad('black',1.)
img = ax[1].imshow(np.fliplr(grid), aspect='auto', cmap=cmap)

for r in [0.25, 0.5, 0.75, 1]:
    M = 100
    xc = np.cos(np.linspace(0, 2 * np.pi, M)) * r
    yc = np.sin(np.linspace(0, 2 * np.pi, M)) * r
    ax[0].plot(xc, yc, color='grey', alpha=0.5, linewidth=0.5, linestyle='dotted')


for angle in [-pi/4, 0, pi/4]:
    line_lx, line_ly = polar_to_cartesian(1, [-pi/4, 3*pi/4])
    line_cx, line_cy = polar_to_cartesian(1, [-pi/2, -3*pi/2])
    line_rx, line_ry = polar_to_cartesian(1, [pi/4, -3*pi/4])
    ax[0].plot(line_lx, line_ly, color='grey', alpha=0.5, linewidth=0.5, linestyle='dotted')
    ax[0].plot(line_cx, line_cy, color='grey', alpha=0.5, linewidth=0.5, linestyle='dotted')
    ax[0].plot(line_rx, line_ry, color='grey', alpha=0.5, linewidth=0.5, linestyle='dotted')
    text_rx, text_ry = polar_to_cartesian(1.1, pi/4)
    text_cx, text_cy = polar_to_cartesian(0.9, pi/2 + 0.1)
    text_lx, text_ly = polar_to_cartesian(1.1, 3*pi/4)

    ax[0].text(text_rx, text_ry, "R", horizontalalignment='center', verticalalignment='center', transform=ax[0].transData, color='grey')
    ax[0].text(text_cx, text_cy, "C", horizontalalignment='center', verticalalignment='center', transform=ax[0].transData, color='grey')
    ax[0].text(text_lx, text_ly, "L", horizontalalignment='center', verticalalignment='center', transform=ax[0].transData, color='grey')

ax[1].hlines(np.arange(2*N - 1) + 0.5, xmin=-0.5, xmax=19.5, color='white', alpha=0.5, linewidth=0.5)
ax[1].vlines(np.arange(2*N - 1) + 0.5, ymin=-0.5, ymax=19.5, color='white', alpha=0.5, linewidth=0.5)

ax[1].text(0.45, 0.05, f"Unique boxes = {unique_boxes}/{(2*N)**2}", color='white', bbox=dict(facecolor='black', edgecolor='white'), transform=ax[1].transAxes)

ax[0].set_xticks([])
ax[1].set_xticks([])
ax[0].set_yticks([])
ax[1].set_yticks([])
ax[0].set_title("Goniometer")
ax[1].set_title("Boxes")

cbar = fig.colorbar(img)
cbar.set_label("Number of samples in box")

fig.savefig("../../papers/kappa/figures/goniometer.pdf")